# Telecom Customer Churn — Retention Analysis

This notebook answers which subscribers are most likely to churn and where a retention budget should be spent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score


In [ ]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_total_charges = int(df['TotalCharges'].isna().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)
df.head()


## Data quality and KPI checks

In [ ]:
print(df.shape)
print('Blank TotalCharges converted to NA:', df['TotalCharges'].isna().sum())
print('Overall churn rate:', round(df['ChurnFlag'].mean()*100, 1), '%')


In [ ]:
contract_summary = (df.groupby('Contract', as_index=False)
    .agg(customers=('customerID','count'), churn_rate=('ChurnFlag','mean'),
         mrr_lost=('MonthlyCharges', lambda x: x[df.loc[x.index, 'ChurnFlag'].eq(1)].sum())))
contract_summary['churn_rate_pct'] = contract_summary['churn_rate'] * 100
contract_summary.sort_values('churn_rate_pct', ascending=False)

## Segmentation

In [ ]:
df['tenure_band'] = pd.cut(df['tenure'], [-1,3,6,12,24,48,72], labels=['0–3','4–6','7–12','13–24','25–48','49–72'])
tenure_summary = df.groupby('tenure_band', observed=False)['ChurnFlag'].mean().mul(100)
tenure_summary.plot(kind='line', marker='o', title='Churn rate by tenure band');
plt.ylabel('Churn rate (%)'); plt.show()

In [ ]:
segment = (df.groupby(['InternetService','TechSupport'], as_index=False)
  .agg(customers=('customerID','count'), churn_rate=('ChurnFlag','mean')))
segment['churn_rate_pct'] = segment['churn_rate'] * 100
segment.sort_values('churn_rate_pct', ascending=False)

## Logistic regression and random forest comparison

In [ ]:
target = df['ChurnFlag']
features = ['tenure','MonthlyCharges','TotalCharges','Contract','InternetService','TechSupport','PaymentMethod','PaperlessBilling']
X_train, X_test, y_train, y_test = train_test_split(df[features], target, test_size=0.2, random_state=42, stratify=target)
num_cols = ['tenure','MonthlyCharges','TotalCharges']
cat_cols = [c for c in features if c not in num_cols]
preprocess = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
logit = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))])
forest = Pipeline([('preprocess', preprocess), ('classifier', RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', min_samples_leaf=5, n_jobs=-1))])
results = []
for name, estimator in [('Logistic regression', logit), ('Random forest', forest)]:
    estimator.fit(X_train, y_train)
    pred_prob = estimator.predict_proba(X_test)[:, 1]
    pred = (pred_prob >= 0.5).astype(int)
    results.append({'model': name, 'roc_auc': roc_auc_score(y_test, pred_prob), 'churn_recall': recall_score(y_test, pred)})
results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df

## Conclusion

The descriptive finding is stable across the analysis: month-to-month and early-tenure customers are the highest-priority retention segments. The models provide a reproducible way to rank customers for outreach, while the dashboard communicates the segment-level decision to a business audience.